# Behavioral-cloning baseline inference

Standalone headless Colab evaluation for the CNN behavior-cloning policy. It loads the validation-selected all-motion checkpoint, restores exact `Lift_D0` configurations from `evals.hdf5`, executes one primitive action per simulator step, and saves synchronized overhead/wrist MP4 rollouts to `MyDrive/baseline/rollouts`.


In [ ]:
%pip install -q h5py imageio imageio-ffmpeg mujoco==3.2.4 robosuite==1.4.1 git+https://github.com/NVlabs/mimicgen.git@72bd767c255545f462e7ccfb2731f2e5d4c1d9bb


In [ ]:
import ctypes
import re
import time
from pathlib import Path

import h5py
import imageio.v2 as imageio
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from google.colab import drive
drive.mount('/content/drive')

BASELINE_DIR = Path('/content/drive/MyDrive/baseline')
MODEL_DIR = BASELINE_DIR / 'models'
ROLLOUT_DIR = BASELINE_DIR / 'rollouts'
ROLLOUT_DIR.mkdir(parents=True, exist_ok=True)
PRIMARY_DIR = Path('/content/drive/MyDrive/primary')
ENV_NAME = 'Lift_D0'
ARCHITECTURE = 'cnn_spatial_softargmax_bounded_all_motion_cosine_v4'
EVAL_GOAL_STRIDE = 25
EVAL_SUBGOAL_BUDGET = 50


def epoch_number(path):
    match = re.search(r'epoch_(\d+)_baseline\.pth$', path.name)
    return int(match.group(1)) if match else -1


def select_checkpoint(model_dir):
    best = model_dir / 'best_baseline.pth'
    if best.exists():
        return best, 'best validation checkpoint'
    latest = model_dir / 'baseline_latest.pth'
    if latest.exists():
        return latest, 'latest checkpoint (fallback)'
    epochs = list(model_dir.glob('epoch_*_baseline.pth'))
    if not epochs:
        raise FileNotFoundError(f'No baseline checkpoints found in {model_dir}')
    checkpoint = max(epochs, key=epoch_number)
    return checkpoint, f'epoch {epoch_number(checkpoint)} (fallback)'


CHECKPOINT, CHECKPOINT_LABEL = select_checkpoint(MODEL_DIR)
eval_candidates = [
    PRIMARY_DIR / 'evals.hdf5',
    Path('/content/data/robot/mimicgen/evals.hdf5'),
]
EVAL_HDF5 = next((path for path in eval_candidates if path.exists()), None)
if EVAL_HDF5 is None:
    raise FileNotFoundError(
        'Could not find evals.hdf5. Put it in /content/drive/MyDrive/primary/.'
    )
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))
print(f'checkpoint: {CHECKPOINT} ({CHECKPOINT_LABEL})')
print(f'exact-reset eval HDF5: {EVAL_HDF5}')


In [ ]:
class SpatialSoftArgmax(nn.Module):
    """Per-channel spatial softmax → expected (x, y) in [-1, 1]."""
    def forward(self, x):
        batch, channels, height, width = x.shape
        weights = F.softmax(x.reshape(batch, channels, -1), dim=-1).reshape(
            batch, channels, height, width
        )
        ys = torch.linspace(-1.0, 1.0, height, device=x.device, dtype=x.dtype)
        xs = torch.linspace(-1.0, 1.0, width, device=x.device, dtype=x.dtype)
        expected_y = (weights.sum(dim=-1) * ys.view(1, 1, height)).sum(dim=-1)
        expected_x = (weights.sum(dim=-2) * xs.view(1, 1, width)).sum(dim=-1)
        return torch.stack((expected_x, expected_y), dim=-1).reshape(batch, channels * 2)


class BaselinePolicy(nn.Module):
    def __init__(self, d_state=9, d_action=7, hidden=1536):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(6, 128, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 384, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )
        self.spatial_softargmax = SpatialSoftArgmax()
        self.action_head = nn.Sequential(
            nn.Linear(384 * 2 + d_state, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, d_action),
        )

    def forward(self, above, wrist, state):
        features = self.spatial_softargmax(self.cnn(torch.cat((above, wrist), dim=1)))
        raw = self.action_head(torch.cat((features, state), dim=-1))
        continuous = torch.tanh(raw[:, :6])
        return torch.cat((continuous, raw[:, 6:7]), dim=-1)


def load_policy(checkpoint, device):
    payload = torch.load(checkpoint, map_location=device, weights_only=True)
    if not isinstance(payload, dict) or 'model_state_dict' not in payload:
        raise ValueError(
            'Checkpoint uses the old weights-only format. Train once with the updated '
            'train_baseline.ipynb so state normalization statistics are saved.'
        )
    if payload.get('architecture') != ARCHITECTURE:
        raise ValueError(
            f"Expected architecture {ARCHITECTURE!r}, got {payload.get('architecture')!r}"
        )
    state_mean = payload['state_mean'].float().cpu()
    state_std = payload['state_std'].float().cpu()
    if state_mean.shape != (9,) or state_std.shape != (9,):
        raise ValueError('Checkpoint state statistics must both have shape (9,)')
    if not torch.isfinite(state_mean).all() or not torch.isfinite(state_std).all() or (state_std <= 0).any():
        raise ValueError('Checkpoint contains invalid state normalization statistics')
    policy = BaselinePolicy().to(device)
    policy.load_state_dict(payload['model_state_dict'])
    policy.eval()
    print(
        f"loaded epoch {payload.get('epoch', '?')} | "
        f"validation loss {payload.get('val_loss', float('nan')):.5f} | "
        f"best epoch {payload.get('best_epoch', '?')}"
    )
    return policy, state_mean, state_std


policy, STATE_MEAN, STATE_STD = load_policy(CHECKPOINT, DEVICE)


In [ ]:
CAM_ABOVE = 'agentview'
CAM_WRIST = 'robot0_eye_in_hand'
IMG_SIZE = 84
VIDEO_SIZE = 256
CONTROL_FREQ = 20

import mujoco
import robosuite as suite
import robosuite.macros as macros
from robosuite.controllers import load_controller_config
import mimicgen
from mimicgen.envs.robosuite.single_arm_env_mg import SingleArmEnv_MG
from robosuite.environments.manipulation.lift import Lift


# The pinned MimicGen revision does not ship Lift_D0, so register the same
# minimal adapter used during collection and primary-model inference.
class Lift_D0(Lift, SingleArmEnv_MG):
    def edit_model_xml(self, xml_str):
        return SingleArmEnv_MG.edit_model_xml(self, xml_str)

    def reward(self, action=None):
        return Lift.reward(self, action=action)

macros.IMAGE_CONVENTION = 'opencv'


def set_geom_groups(env, collision=False, visual=True):
    context = env.sim._render_context_offscreen
    if context is None:
        return
    address = context.vopt.geomgroup.__array_interface__['data'][0]
    groups = (ctypes.c_uint8 * 6).from_address(address)
    for index in range(6):
        groups[index] = 0
    groups[0] = int(collision)
    groups[1] = int(visual)


def build_robot_state(obs):
    position = torch.from_numpy(obs['robot0_eef_pos']).float()
    quaternion = torch.from_numpy(obs['robot0_eef_quat']).float()
    quaternion = quaternion / quaternion.norm().clamp_min(1e-8)
    gripper = torch.from_numpy(obs['robot0_gripper_qpos']).float()
    state = torch.cat((position, quaternion, gripper))
    return (state - STATE_MEAN) / STATE_STD


def obs_to_tensors(obs):
    above = torch.from_numpy(obs[f'{CAM_ABOVE}_image']).float().permute(2, 0, 1) / 255.0
    wrist = torch.from_numpy(obs[f'{CAM_WRIST}_image']).float().permute(2, 0, 1) / 255.0
    state = build_robot_state(obs)
    return above[None].to(DEVICE), wrist[None].to(DEVICE), state[None].to(DEVICE)


@torch.inference_mode()
def action_from_policy(obs):
    above, wrist, state = obs_to_tensors(obs)
    output = policy(above, wrist, state).squeeze(0).float().cpu().numpy()
    action = np.clip(output, -1.0, 1.0)
    action[6] = 1.0 if output[6] >= 0 else -1.0
    return action


def make_env(max_steps):
    controller = load_controller_config(default_controller='OSC_POSE')
    env = suite.make(
        ENV_NAME, robots='Panda', controller_configs=controller,
        has_renderer=False, has_offscreen_renderer=True,
        render_collision_mesh=False, render_visual_mesh=True,
        use_camera_obs=True, use_object_obs=True,
        camera_names=[CAM_ABOVE, CAM_WRIST],
        camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
        control_freq=CONTROL_FREQ, horizon=max_steps, ignore_done=True,
    )
    set_geom_groups(env)
    return env


def decode_string(raw):
    return raw.decode('utf-8') if isinstance(raw, bytes) else str(raw)


def load_eval_resets(path, demo_name=None):
    names, initial_states, model_xmls, rollout_budgets = [], [], [], []
    with h5py.File(path, 'r') as file:
        collected_env = file['data'].attrs.get('env')
        if collected_env is None or decode_string(collected_env) != ENV_NAME:
            raise ValueError(
                f'Expected {ENV_NAME} evals, but {path} contains '
                f'{None if collected_env is None else decode_string(collected_env)!r}'
            )
        collected_mujoco = file['data'].attrs.get('mujoco_version')
        if collected_mujoco is not None and decode_string(collected_mujoco) != mujoco.__version__:
            print(f'WARNING: evals use MuJoCo {decode_string(collected_mujoco)}; inference uses {mujoco.__version__}')
        if demo_name is not None:
            demo_names = [demo_name]
        elif 'mask' in file and 'eval' in file['mask']:
            demo_names = [decode_string(raw) for raw in file['mask/eval'][:]]
        else:
            demo_names = sorted(file['data'].keys())
        for name in demo_names:
            demo = file[f'data/{name}']
            initial_index = int(demo.attrs.get('initial_index', 0))
            if initial_index != 0 or 'model_file' not in demo.attrs:
                raise ValueError(f'{name} has invalid exact-reset metadata')
            final_index = int(demo['states'].shape[0] - 1)
            if int(demo.attrs.get('goal_index', -1)) != final_index:
                raise ValueError(f'{name} does not mark its last frame as the final goal')
            num_goal_segments = len(range(EVAL_GOAL_STRIDE, final_index, EVAL_GOAL_STRIDE)) + 1
            names.append(name)
            initial_states.append(np.asarray(demo['states'][initial_index]).copy())
            model_xmls.append(decode_string(demo.attrs['model_file']))
            rollout_budgets.append(num_goal_segments * EVAL_SUBGOAL_BUDGET)
    print(f'exact baseline eval bank: {len(names)} initial configurations')
    return names, initial_states, model_xmls, rollout_budgets


def reset_to_eval_state(env, model_xml, initial_state):
    env.reset()
    xml = env.edit_model_xml(model_xml)
    env.reset_from_xml_string(xml)
    env.sim.reset()
    env.sim.set_state_from_flattened(initial_state)
    env.sim.forward()
    set_geom_groups(env)
    return env._get_observations(force_update=True)


def render_frame(env):
    views = []
    for camera in (CAM_ABOVE, CAM_WRIST):
        frame = env.sim.render(height=VIDEO_SIZE, width=VIDEO_SIZE, camera_name=camera)[::-1]
        views.append(np.asarray(frame, dtype=np.uint8))
    return np.concatenate(views, axis=1)


def run_rollout(env, name, initial_state, model_xml, max_steps, video_path):
    video_path.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(video_path), fps=CONTROL_FREQ, codec='libx264')
    try:
        obs = reset_to_eval_state(env, model_xml, initial_state)
        writer.append_data(render_frame(env))
        success = False
        for step in range(1, max_steps + 1):
            action = action_from_policy(obs)
            obs, _, done, _ = env.step(action)
            set_geom_groups(env)
            writer.append_data(render_frame(env))
            success = bool(env._check_success())
            if done or success:
                break
        return success, step
    finally:
        writer.close()


## Run exact-reset headless rollouts

The baseline receives only its current two camera views and normalized robot state. The eval HDF5 is used to restore the same initial configurations and assign the same total primitive-step budget as the segmented primary evaluation. Intermediate goal observations and evaluation actions are never supplied to the baseline.


In [ ]:
NUM_ROLLOUTS = 5
EVAL_DEMO = None
EVAL_SEED = 42

names, initial_states, model_xmls, rollout_budgets = load_eval_resets(EVAL_HDF5, EVAL_DEMO)
rng = np.random.default_rng(EVAL_SEED)
sampled_eval_indices = rng.choice(
    len(names), size=NUM_ROLLOUTS, replace=NUM_ROLLOUTS > len(names)
)
env = make_env(max(rollout_budgets[index] for index in sampled_eval_indices))
successes = 0
start_time = time.perf_counter()
try:
    for rollout_index in range(NUM_ROLLOUTS):
        eval_index = int(sampled_eval_indices[rollout_index])
        name = names[eval_index]
        max_steps = rollout_budgets[eval_index]
        pending_video_path = ROLLOUT_DIR / f'{CHECKPOINT.stem}_{name}_bc_rollout_pending.mp4'
        success, steps = run_rollout(
            env, name, initial_states[eval_index], model_xmls[eval_index],
            max_steps, pending_video_path,
        )
        result_label = 'pass' if success else 'fail'
        video_path = pending_video_path.with_name(
            pending_video_path.name.replace('_pending.mp4', f'_{result_label}.mp4')
        )
        video_path.unlink(missing_ok=True)
        pending_video_path.replace(video_path)
        successes += int(success)
        print(
            f'rollout {rollout_index + 1}/{NUM_ROLLOUTS}: '
            f'{"success" if success else "fail"} after {steps}/{max_steps} steps'
        )
        print(f'video: {video_path}')
finally:
    env.close()
elapsed = time.perf_counter() - start_time
print(f'success rate: {successes}/{NUM_ROLLOUTS} = {100 * successes / NUM_ROLLOUTS:.1f}%')
print(f'total rollout time: {elapsed:.2f}s')


## Benchmark policy inference

Measures the neural-policy forward pass only; environment stepping and video encoding are excluded.


In [ ]:
dummy_above = torch.rand(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
dummy_wrist = torch.rand(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
dummy_state = torch.zeros(1, 9, device=DEVICE)
for _ in range(20):
    with torch.inference_mode():
        policy(dummy_above, dummy_wrist, dummy_state)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
start = time.perf_counter()
iterations = 200
for _ in range(iterations):
    with torch.inference_mode():
        policy(dummy_above, dummy_wrist, dummy_state)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
seconds = time.perf_counter() - start
print(f'policy latency: {1000 * seconds / iterations:.3f} ms/action')
print(f'policy throughput: {iterations / seconds:.1f} actions/s')
